# ЛР3. Клиент

In [ ]:
import xmlrpc.client
import pickle
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline

def display_results(original, processed, title):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.title("Оригинал")
    cmap = 'gray' if len(original.shape) == 2 else None
    plt.imshow(original, cmap=cmap)
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.title(title)
    cmap = 'gray' if len(processed.shape) == 2 else None
    plt.imshow(processed, cmap=cmap)
    plt.axis('off')
    plt.show()

server = xmlrpc.client.ServerProxy("http://localhost:8010", allow_none=True)
stats = xmlrpc.client.ServerProxy("http://localhost:8009", allow_none=True)

print("Тест базовых функций:")
print("Ping:", server.ping())
print("Now:", server.now())
print("Type:", server.show_type(2))
print("Sum:", server.test_sum(2, 3))
print("Pow:", server.test_pow(2, 3))

print("\nПроверка чёрного списка:")
test_cases = [
    ('Иванов', 'Иван', 'Иванович', '22.03.1989'),
    ('иванов', 'иван', 'иванович', '22.03.1989'),
    ('Иваноа', 'Иван', 'Иванович', '22.03.1989'),
    ('Сидоров', 'Пётр', 'Петрович', '01.01.1990')
]
for args in test_cases:
    print(args, "=>", server.black_list_check(*args))

print("\nОбработка изображений:")
img_color = np.array(Image.open('Jellyfish.jpg'))
img_mono = np.array(Image.open('11.bmp'))
pimg_color = xmlrpc.client.Binary(pickle.dumps(img_color))
pimg_mono = xmlrpc.client.Binary(pickle.dumps(img_mono))
threshold = 40

print("Инверсия:")
inv_color = pickle.loads(server.color_inversion(pimg_color).data)
inv_mono = pickle.loads(server.color_inversion(pimg_mono).data)
display_results(img_color, inv_color, "Инверсия (цветное)")
display_results(img_mono, inv_mono, "Инверсия (ч/б)")

print("Бинаризация:")
bw_color = pickle.loads(server.binarize_image(pimg_color, threshold).data)
bw_mono = pickle.loads(server.binarize_image(pimg_mono, threshold).data)
display_results(img_color, bw_color, f"Бинаризация (порог {threshold})")
display_results(img_mono, bw_mono, f"Бинаризация (порог {threshold})")

print("Разворот:")
flip_color = pickle.loads(server.flip_vertical(pimg_color).data)
flip_mono = pickle.loads(server.flip_vertical(pimg_mono).data)
display_results(img_color, flip_color, "Разворот")
display_results(img_mono, flip_mono, "Разворот")

print("Облачность:")
print(f"Цветное: {server.cloudiness_percent(pimg_color, threshold):.2f}%")
print(f"Ч/б: {server.cloudiness_percent(pimg_mono, threshold):.2f}%")

print("\nТест Rate Limiting (6 вызовов):")
for i in range(6):
    print(f"Вызов {i+1}:", server.ping())

print("\nЖурнал событий:")
events = stats.get_events('', '', '', '', '')
print(f"Всего событий: {len(events)}")
print("Последние 10:")
for e in events[-10:]:
    print(e)

print("\nФильтрация по типу 'test_sum':")
sum_events = stats.get_events('test_sum', '', '', '', '')
for e in sum_events:
    print(e)

print("\nОбщая статистика:")
stats_data = stats.get_stats()
print(f"Всего: {stats_data['count']}")
print(f"Средняя длительность: {stats_data['avg_duration']:.6f} сек")
print(f"Мин: {stats_data['min_duration']:.6f} сек")
print(f"Макс: {stats_data['max_duration']:.6f} сек")
